# LiCSBAS Jakarta factor-5 resolution sensitivity — gated multi-prime loop handling

**Frame:** `098A_09673_121312` (ascending)  
**Period:** 2020-01-01 to 2023-12-31  
**Pinned COMET-LiCSBAS commit:** `a145e6c4217b29eb7348ea95630454a099809098`  
**Verified against the post-migration downloader:** 28 July 2026

This self-contained notebook tests whether additional spatial averaging can produce a scientifically usable network after the factor-2 multi-prime run failed loop closure. Version 7 retains LiCSBAS's recommended `--multi_prime` option, explicitly keeps the standard 1.5-radian loop threshold, archives step-12 diagnostics automatically, and applies a fail-loud network-survival gate before inversion.

**Resolution policy:** this version runs `NLOOK = 5` at approximately 553 m using multi-prime loop-bias handling. Outputs use `_ml5_mp`; they cannot overwrite the failed factor-2 `_ml2`/`_ml2_mp` diagnostics or the factor-10 baseline. The standard 1.5-radian loop threshold is not relaxed.

Important operating rules:

1. Colab preserves output text from dead sessions. A cell that appears finished but streams no new output did not rerun.
2. Keep the tab active during the download.
3. Use **Runtime → Run all**. The notebook restores the verified 609-pair Drive archive and any verified factor-5 checkpoint automatically; otherwise it downloads, validates, and checkpoints the required data.
4. Never combine a velocity file from one inversion with `vstd` from another.
5. Every external command below checks its return code and required outputs before reporting success.
6. Processing stops before step 9 unless every interferogram folder contains both unwrapped phase (`UNW`) and coherence (`CC`).
7. A CPU/standard Colab runtime is sufficient; this workflow does not use a GPU.
8. No between-cell repair, diagnostic-packaging, or renaming code is required: resumption, exact-date labels, `ml5_mp` tags, resolution diagnostics, network gates, checksums, QC summaries, and archives are built in.
9. The factor-2 `_ml2` and `_ml2_mp` outputs remain preserved as failed diagnostics and are never overwritten.


In [ ]:
# 1 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2 — Install the pinned COMET fork and dependencies
import os, sys, subprocess, pathlib

LICSBAS_REPO = pathlib.Path('/content/LiCSBAS')
LICSBAS_COMMIT = 'a145e6c4217b29eb7348ea95630454a099809098'
if not LICSBAS_REPO.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/comet-licsar/LiCSBAS.git', str(LICSBAS_REPO)],
        check=True,
    )
subprocess.run(['git', '-C', str(LICSBAS_REPO), 'fetch', '--all', '--tags'], check=True)
subprocess.run(['git', '-C', str(LICSBAS_REPO), 'checkout', '--detach', LICSBAS_COMMIT], check=True)
actual_commit = subprocess.check_output(
    ['git', '-C', str(LICSBAS_REPO), 'rev-parse', 'HEAD'], text=True
).strip()
assert actual_commit == LICSBAS_COMMIT, (actual_commit, LICSBAS_COMMIT)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'h5py', 'statsmodels', 'astropy', 'cmcrameri', 'bottleneck', 'rasterio'
], check=True)
requirements = LICSBAS_REPO / 'requirements.txt'
if requirements.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements)], check=True)
print('LiCSBAS commit:', actual_commit)


In [ ]:
# 3 — Configuration and command runner
import json, hashlib, datetime, glob, re, math, shutil

FRAME = '098A_09673_121312'
START = '20200101'
END = '20231231'
TEST_START = '20221205'
TEST_END = '20230122'
NLOOK = 5
GEOCML_NAME = f'GEOCml{NLOOK}'
TS_NAME = f'TS_{GEOCML_NAME}_mp'
RUN_TAG = f'ml{NLOOK}_mp'
DATE_TAG = f'{START}_{END}'
PERIOD_LABEL = f'{START[:4]}-{END[:4]}'
RUN_ROLE = 'intermediate_resolution_sensitivity_multi_prime'
LOOP_MODE = 'multi_prime'
LOOP_THRESHOLD_RAD = 1.5
MAX_LOOP_BAD_FRACTION = 0.25
MIN_RETAINED_IMAGE_FRACTION = 0.75
MIN_TEMPORAL_SPAN_FRACTION = 0.90
EXPECTED_COMPLETE_PAIRS = 609
MIN_COMPLETE_PAIRS = 20
RESUME_FROM_VERIFIED_ARCHIVE = True
WORK = pathlib.Path('/content/insar')
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/InSAR_Risk_Zoning_Project')
ARCHIVE = DRIVE_OUT / 'LiCSBAS_archive_2020_2023'
BIN = LICSBAS_REPO / 'bin'
LIB = LICSBAS_REPO / 'LiCSBAS_lib'
WORK.mkdir(parents=True, exist_ok=True)
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
ARCHIVE.mkdir(parents=True, exist_ok=True)
GEOC_ARCHIVE = ARCHIVE / 'GEOC_20200101_20231231_verified_v4.tar.gz'
GEOC_ARCHIVE_SHA = ARCHIVE / 'GEOC_20200101_20231231_verified_v4.sha256'
GEOC_INVENTORY = ARCHIVE / 'GEOC_20200101_20231231_verified_v4_inventory.json'
GEOCML_ARCHIVE = ARCHIVE / f'{GEOCML_NAME}_{DATE_TAG}_checkpoint.tar.gz'
GEOCML_ARCHIVE_SHA = ARCHIVE / f'{GEOCML_NAME}_{DATE_TAG}_checkpoint.sha256'
GEOCML_INVENTORY = ARCHIVE / f'{GEOCML_NAME}_{DATE_TAG}_inventory.json'
QC_SUMMARY = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_qc_summary.json'

run_env = os.environ.copy()
run_env['PYTHONPATH'] = str(LIB)

def resolve_script(role, preferred, fragments):
    exact = BIN / preferred
    if exact.exists():
        return exact
    candidates = sorted(
        p for p in BIN.glob('LiCSBAS*.py')
        if all(fragment.lower() in p.name.lower() for fragment in fragments)
    )
    if len(candidates) != 1:
        raise RuntimeError(f'Cannot resolve {role}: {candidates}')
    return candidates[0]

SCRIPTS = {
    'download': resolve_script('download', 'LiCSBAS01_get_geotiff.py', ('01', 'get_geo')),
    'multilook': resolve_script('multilook', 'LiCSBAS02_ml_prep.py', ('02', 'ml')),
    'check': resolve_script('check', 'LiCSBAS11_check_unw.py', ('11', 'check_unw')),
    'loop': resolve_script('loop', 'LiCSBAS12_loop_closure.py', ('12', 'loop')),
    'invert': resolve_script('invert', 'LiCSBAS13_sb_inv.py', ('13', 'sb_inv')),
    'vstd': resolve_script('vstd', 'LiCSBAS14_vel_std.py', ('14', 'vel_std')),
    'geotiff': resolve_script('GeoTIFF export', 'LiCSBAS_flt2geotiff.py', ('flt2geotiff',)),
}

def run_script(role, args, required=()):
    cmd = [sys.executable, str(SCRIPTS[role]), *map(str, args)]
    print('RUN:', ' '.join(cmd), flush=True)
    completed = subprocess.run(cmd, cwd=WORK, env=run_env)
    if completed.returncode != 0:
        raise RuntimeError(f'{role} failed with return code {completed.returncode}')
    missing = [str(p) for p in required if not pathlib.Path(p).exists()]
    if missing:
        raise RuntimeError(f'{role} completed but required output is missing: {missing}')
    return completed

PAIR_RE = re.compile(r'^\d{8}_\d{8}$')

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def _has_product(folder, kind):
    patterns = (f'*.geo.{kind}.tif', f'*.{kind}')
    return any(
        path.is_file() and path.stat().st_size > 0
        for pattern in patterns for path in folder.glob(pattern)
    )

def inventory_pairs(root):
    root = pathlib.Path(root)
    pair_dirs = sorted(
        path for path in root.iterdir()
        if path.is_dir() and PAIR_RE.fullmatch(path.name)
    ) if root.exists() else []
    missing_unw = [path.name for path in pair_dirs if not _has_product(path, 'unw')]
    missing_cc = [path.name for path in pair_dirs if not _has_product(path, 'cc')]
    incomplete = sorted(set(missing_unw) | set(missing_cc))
    return {
        'root': str(root),
        'pair_folders': len(pair_dirs),
        'complete_pairs': len(pair_dirs) - len(incomplete),
        'missing_unw': missing_unw,
        'missing_cc': missing_cc,
        'incomplete_pairs': incomplete,
    }

def require_complete_pairs(root, label, min_pairs=1):
    inventory = inventory_pairs(root)
    print(label, json.dumps(inventory, indent=2))
    if inventory['pair_folders'] < min_pairs:
        raise RuntimeError(
            f'{label} has only {inventory["pair_folders"]} pair folders; '
            f'at least {min_pairs} are required.'
        )
    if inventory['incomplete_pairs']:
        raise RuntimeError(
            f'{label} contains {len(inventory["incomplete_pairs"])} incomplete pairs. '
            'Do not continue to LiCSBAS step 11.'
        )
    return inventory

print(json.dumps({k: str(v) for k, v in SCRIPTS.items()}, indent=2))


In [ ]:
# 4 — Validate every resolved script and inspect step-02 flags
for role, script in SCRIPTS.items():
    help_run = subprocess.run(
        [sys.executable, str(script), '-h'], cwd=WORK, env=run_env,
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    if help_run.returncode not in (0, 1):
        raise RuntimeError(f'{role} help check failed ({help_run.returncode})')
    print(f'--- {role}: {script.name} ---')
    print('\n'.join(help_run.stdout.splitlines()[:12]))

step02_help = subprocess.run(
    [sys.executable, str(SCRIPTS['multilook']), '-h'], cwd=WORK, env=run_env,
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
).stdout
for required_flag in ('-i', '-o', '-n'):
    if required_flag not in step02_help:
        raise RuntimeError(f'Step 02 does not advertise required flag {required_flag}')
print('All command/help checks passed.')


## Automatic recovery after a lost Colab session

When a checksum-verified archive from this notebook exists in Drive, the next cell restores it automatically. Otherwise it starts a fresh download. The older incomplete 5.96 GB archive is deliberately ignored.


In [ ]:
# 5 — Restore only a checksum-verified archive produced by this notebook
RESTORED_GEOC = False
if RESUME_FROM_VERIFIED_ARCHIVE and GEOC_ARCHIVE.exists() and GEOC_ARCHIVE_SHA.exists():
    expected_sha = GEOC_ARCHIVE_SHA.read_text(encoding='utf-8').strip().split()[0]
    actual_sha = sha256_file(GEOC_ARCHIVE)
    if actual_sha != expected_sha:
        raise RuntimeError('The Drive GEOC archive checksum does not match; refusing to restore it.')
    if not (WORK / 'GEOC').exists():
        subprocess.run(['tar', '-xzf', str(GEOC_ARCHIVE), '-C', str(WORK)], check=True)
    restored_inventory = require_complete_pairs(WORK / 'GEOC', 'Restored GEOC', MIN_COMPLETE_PAIRS)
    if restored_inventory['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
        raise RuntimeError(f'Restored GEOC has {restored_inventory["complete_pairs"]} complete pairs; expected {EXPECTED_COMPLETE_PAIRS}.')
    RESTORED_GEOC = True
    print('Restored verified GEOC archive:', GEOC_ARCHIVE)
else:
    print('Verified 6.40 GB archive not found. A fresh migration-safe download will run.')
print('The older incomplete 5.96 GB archive is intentionally ignored.')


In [ ]:
# 6 — Test the migrated service, then download and validate the full GEOC stack
if not RESTORED_GEOC:
    existing_inventory = inventory_pairs(WORK / 'GEOC')
    if existing_inventory['pair_folders']:
        print(
            f'Found {existing_inventory["pair_folders"]} partial/existing pair folders. '
            'Skipping the clean preflight gate and resuming the full download.'
        )
    else:
        print(f'Preflight download: {TEST_START} to {TEST_END}')
        run_script(
            'download', ['-f', FRAME, '-s', TEST_START, '-e', TEST_END, '--n_para', '4'],
            required=[WORK / 'GEOC'],
        )
        require_complete_pairs(WORK / 'GEOC', 'Preflight GEOC', min_pairs=1)
    print(f'Full download/resume: {START} to {END}')
    run_script(
        'download', ['-f', FRAME, '-s', START, '-e', END, '--n_para', '4'],
        required=[WORK / 'GEOC'],
    )

geoc_inventory = require_complete_pairs(WORK / 'GEOC', 'Full GEOC', MIN_COMPLETE_PAIRS)
if geoc_inventory['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'Full GEOC has {geoc_inventory["complete_pairs"]} complete pairs; expected {EXPECTED_COMPLETE_PAIRS}.')
geoc_inventory.update({
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'frame': FRAME,
    'start': START,
    'end': END,
    'licsbas_commit': actual_commit,
})
GEOC_INVENTORY.write_text(json.dumps(geoc_inventory, indent=2), encoding='utf-8')
print('GEOC inventory saved:', GEOC_INVENTORY)


In [ ]:
# 7 — Back up the validated GEOC stack and verify its contents and checksum
if not RESTORED_GEOC:
    partial_archive = GEOC_ARCHIVE.with_suffix(GEOC_ARCHIVE.suffix + '.partial')
    backup = subprocess.run(
        ['tar', '-czf', str(partial_archive), '-C', str(WORK), 'GEOC']
    )
    if backup.returncode != 0 or not partial_archive.exists() or partial_archive.stat().st_size == 0:
        raise RuntimeError('GEOC backup failed; do not continue')
    os.replace(partial_archive, GEOC_ARCHIVE)
listing = subprocess.run(
    ['tar', '-tzf', str(GEOC_ARCHIVE)], check=True,
    text=True, stdout=subprocess.PIPE,
).stdout.splitlines()
archive_unw = {pathlib.PurePosixPath(item).parts[1] for item in listing if item.endswith('.geo.unw.tif') and len(pathlib.PurePosixPath(item).parts) >= 3}
archive_cc = {pathlib.PurePosixPath(item).parts[1] for item in listing if item.endswith('.geo.cc.tif') and len(pathlib.PurePosixPath(item).parts) >= 3}
if archive_unw != archive_cc or len(archive_unw) != geoc_inventory['complete_pairs']:
    raise RuntimeError(
        f'Archive pair mismatch: UNW={len(archive_unw)}, CC={len(archive_cc)}, '
        f'expected={geoc_inventory["complete_pairs"]}'
    )
archive_sha = sha256_file(GEOC_ARCHIVE)
GEOC_ARCHIVE_SHA.write_text(f'{archive_sha}  {GEOC_ARCHIVE.name}\n', encoding='utf-8')
print(f'VERIFIED backup: {GEOC_ARCHIVE} ({GEOC_ARCHIVE.stat().st_size / 1e9:.2f} GB, {len(archive_unw):,} complete pairs)')
print('SHA-256:', archive_sha)


In [ ]:
# 8 — Step 02: restore or create the factor-tagged multilook checkpoint
import rasterio

GEOCML_RESTORED = False
geocml_root = WORK / GEOCML_NAME
if not geocml_root.exists() and GEOCML_ARCHIVE.exists() and GEOCML_ARCHIVE_SHA.exists():
    expected_ml_sha = GEOCML_ARCHIVE_SHA.read_text(encoding='utf-8').strip().split()[0]
    actual_ml_sha = sha256_file(GEOCML_ARCHIVE)
    if actual_ml_sha != expected_ml_sha:
        raise RuntimeError('The multilook checkpoint checksum does not match; refusing to restore it.')
    subprocess.run(['tar', '-xzf', str(GEOCML_ARCHIVE), '-C', str(WORK)], check=True)
    GEOCML_RESTORED = True
    print('Restored verified multilook checkpoint:', GEOCML_ARCHIVE)

existing_ml = inventory_pairs(geocml_root)
if existing_ml['pair_folders'] and existing_ml['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
    print('Removing an incomplete in-runtime multilook directory before rebuilding it.')
    shutil.rmtree(geocml_root)
    GEOCML_RESTORED = False

if not geocml_root.exists():
    run_script(
        'multilook', ['-i', 'GEOC', '-o', GEOCML_NAME, '-n', NLOOK],
        required=[geocml_root],
    )
else:
    print(f'{GEOCML_NAME} already exists; validating it instead of repeating step 02.')

geocml_inventory = require_complete_pairs(geocml_root, GEOCML_NAME, MIN_COMPLETE_PAIRS)
if geocml_inventory['complete_pairs'] != EXPECTED_COMPLETE_PAIRS:
    raise RuntimeError(f'{GEOCML_NAME} has {geocml_inventory["complete_pairs"]} complete pairs; expected {EXPECTED_COMPLETE_PAIRS}.')

source_unw = next((WORK / 'GEOC').glob('*/*.geo.unw.tif'))
with rasterio.open(source_unw) as src:
    centre_lat = (src.bounds.top + src.bounds.bottom) / 2
    source_x_m = abs(src.res[0]) * 111320.0 * math.cos(math.radians(centre_lat))
    source_y_m = abs(src.res[1]) * 111132.0
    source_shape = [src.height, src.width]
    source_res_deg = [abs(src.res[0]), abs(src.res[1])]

dem_values = {}
for line in (geocml_root / 'EQA.dem_par').read_text(encoding='utf-8', errors='replace').splitlines():
    fields = line.replace(':', ' ').split()
    if len(fields) >= 2 and fields[0] in {'width', 'nlines', 'post_lat', 'post_lon'}:
        dem_values[fields[0]] = fields[1]
resolution_summary = {
    'source_file': str(source_unw),
    'source_shape': source_shape,
    'source_resolution_degrees': source_res_deg,
    'source_resolution_metres_approx': [source_x_m, source_y_m],
    'multilook_factor': NLOOK,
    'effective_resolution_metres_approx': [source_x_m * NLOOK, source_y_m * NLOOK],
    'multilooked_parameters': dem_values,
}
geocml_inventory.update({
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'frame': FRAME, 'start': START, 'end': END, 'run_tag': RUN_TAG,
    'resolution': resolution_summary,
})
GEOCML_INVENTORY.write_text(json.dumps(geocml_inventory, indent=2), encoding='utf-8')
print(json.dumps(resolution_summary, indent=2))

if not GEOCML_RESTORED:
    partial_ml = GEOCML_ARCHIVE.with_suffix(GEOCML_ARCHIVE.suffix + '.partial')
    checkpoint_run = subprocess.run(['tar', '-czf', str(partial_ml), '-C', str(WORK), GEOCML_NAME])
    if checkpoint_run.returncode != 0 or not partial_ml.exists() or partial_ml.stat().st_size == 0:
        raise RuntimeError('Multilook checkpoint creation failed; do not continue.')
    os.replace(partial_ml, GEOCML_ARCHIVE)
checkpoint_sha = sha256_file(GEOCML_ARCHIVE)
GEOCML_ARCHIVE_SHA.write_text(f'{checkpoint_sha}  {GEOCML_ARCHIVE.name}\n', encoding='utf-8')
subprocess.run(['tar', '-tzf', str(GEOCML_ARCHIVE)], check=True, stdout=subprocess.DEVNULL)
print(f'VERIFIED {RUN_TAG} checkpoint: {GEOCML_ARCHIVE} ({GEOCML_ARCHIVE.stat().st_size / 1e9:.2f} GB)')
print('Checkpoint SHA-256:', checkpoint_sha)


In [ ]:
# 9 — Steps 11 and 12: quality check, multi-prime loop closure, and network gate
run_script('check', ['-d', GEOCML_NAME, '-t', TS_NAME], required=[WORK / TS_NAME])
run_script(
    'loop', ['-d', GEOCML_NAME, '-t', TS_NAME, '-l', str(LOOP_THRESHOLD_RAD), '--multi_prime'],
    required=[WORK / TS_NAME / 'info' / '12bad_ifg.txt'],
)

def data_lines(path):
    path = pathlib.Path(path)
    if not path.exists():
        return []
    return [line.strip() for line in path.read_text(encoding='utf-8', errors='replace').splitlines() if line.strip() and not line.lstrip().startswith('#')]

info_dir = WORK / TS_NAME / 'info'
bad11 = data_lines(info_dir / '11bad_ifg.txt')
bad12 = data_lines(info_dir / '12bad_ifg.txt')
removed12 = data_lines(info_dir / '12removed_image.txt')
all_images = sorted({date for path in (WORK / GEOCML_NAME).iterdir() if path.is_dir() and PAIR_RE.fullmatch(path.name) for date in path.name.split('_')})
retained_images = sorted(set(all_images) - set(removed12))
retained_ifgs = EXPECTED_COMPLETE_PAIRS - len(bad11) - len(bad12)
bad_fraction = (len(bad11) + len(bad12)) / EXPECTED_COMPLETE_PAIRS
image_fraction = len(retained_images) / len(all_images)
requested_days = (datetime.datetime.strptime(END, '%Y%m%d') - datetime.datetime.strptime(START, '%Y%m%d')).days
if len(retained_images) >= 2:
    retained_days = (datetime.datetime.strptime(retained_images[-1], '%Y%m%d') - datetime.datetime.strptime(retained_images[0], '%Y%m%d')).days
    temporal_span_fraction = retained_days / requested_days
else:
    temporal_span_fraction = 0.0
network_gate = {
    'loop_mode': LOOP_MODE,
    'loop_threshold_rad': LOOP_THRESHOLD_RAD,
    'total_interferograms': EXPECTED_COMPLETE_PAIRS,
    'step11_bad_interferograms': len(bad11),
    'step12_bad_interferograms': len(bad12),
    'retained_interferograms': retained_ifgs,
    'bad_interferogram_fraction': bad_fraction,
    'total_images': len(all_images),
    'retained_images': len(retained_images),
    'retained_image_fraction': image_fraction,
    'retained_first_image': retained_images[0] if retained_images else None,
    'retained_last_image': retained_images[-1] if retained_images else None,
    'temporal_span_fraction': temporal_span_fraction,
    'thresholds': {
        'max_loop_bad_fraction': MAX_LOOP_BAD_FRACTION,
        'min_retained_image_fraction': MIN_RETAINED_IMAGE_FRACTION,
        'min_temporal_span_fraction': MIN_TEMPORAL_SPAN_FRACTION,
    },
}
NETWORK_GATE_PATH = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_network_gate.json'
NETWORK_GATE_PATH.write_text(json.dumps(network_gate, indent=2), encoding='utf-8')
NETWORK_GATE_WORK_PATH = info_dir / f'{RUN_TAG}_network_gate.json'
NETWORK_GATE_WORK_PATH.write_text(json.dumps(network_gate, indent=2), encoding='utf-8')
print(json.dumps(network_gate, indent=2))
gate_failures = []
if bad_fraction > MAX_LOOP_BAD_FRACTION:
    gate_failures.append(f'loop closure rejected {bad_fraction:.1%} of interferograms')
if image_fraction < MIN_RETAINED_IMAGE_FRACTION:
    gate_failures.append(f'only {image_fraction:.1%} of acquisition images remain')
if temporal_span_fraction < MIN_TEMPORAL_SPAN_FRACTION:
    gate_failures.append(f'network spans only {temporal_span_fraction:.1%} of the requested period')
DIAGNOSTIC_ARCHIVE = ARCHIVE / f'{TS_NAME}_step12_diagnostics_{DATE_TAG}.tar.gz'
DIAGNOSTIC_ARCHIVE_SHA = ARCHIVE / f'{TS_NAME}_step12_diagnostics_{DATE_TAG}.sha256'
diagnostic_items = [
    f'{TS_NAME}/info',
    f'{TS_NAME}/network',
    f'{TS_NAME}/12loop/loop_info.txt',
    f'{TS_NAME}/12loop/loop_ph_rms_masked.png',
    f'{TS_NAME}/12loop/bad_ifg_loop.txt',
    f'{TS_NAME}/12loop/bad_ifg_loopref.txt',
    f'{TS_NAME}/12loop/bad_ifg_noref.txt',
]
existing_diagnostic_items = [item for item in diagnostic_items if (WORK / item).exists()]
partial_diagnostic = DIAGNOSTIC_ARCHIVE.with_suffix(DIAGNOSTIC_ARCHIVE.suffix + '.partial')
diagnostic_run = subprocess.run(
    ['tar', '-czf', str(partial_diagnostic), '-C', str(WORK), *existing_diagnostic_items]
)
if diagnostic_run.returncode != 0 or not partial_diagnostic.exists() or partial_diagnostic.stat().st_size == 0:
    raise RuntimeError('Step-12 diagnostic archive failed; do not continue.')
os.replace(partial_diagnostic, DIAGNOSTIC_ARCHIVE)
subprocess.run(['tar', '-tzf', str(DIAGNOSTIC_ARCHIVE)], check=True, stdout=subprocess.DEVNULL)
diagnostic_sha = sha256_file(DIAGNOSTIC_ARCHIVE)
DIAGNOSTIC_ARCHIVE_SHA.write_text(f'{diagnostic_sha}  {DIAGNOSTIC_ARCHIVE.name}\n', encoding='utf-8')
print(f'VERIFIED step-12 diagnostics: {DIAGNOSTIC_ARCHIVE} ({DIAGNOSTIC_ARCHIVE.stat().st_size / 1e6:.1f} MB)')
print('Diagnostic SHA-256:', diagnostic_sha)

if gate_failures:
    raise RuntimeError('NETWORK SURVIVAL GATE FAILED: ' + '; '.join(gate_failures) + '. Diagnostics are safely archived in Drive; do not continue to inversion.')
print('NETWORK SURVIVAL GATE PASSED — safe to continue to step 13.')


In [ ]:
# 10 — Steps 13 and 14: SBAS inversion and velocity uncertainty
run_script(
    'invert', ['-d', GEOCML_NAME, '-t', TS_NAME, '--mem_size', '4000'],
    required=[WORK / TS_NAME / 'cum.h5', WORK / TS_NAME / 'results' / 'vel'],
)
run_script(
    'vstd', ['-t', TS_NAME],
    required=[WORK / TS_NAME / 'results' / 'vstd'],
)


In [ ]:
# 11 — Export the matched four-layer GeoTIFF stack with exact dates and factor label
results_dir = WORK / TS_NAME / 'results'
dem_par = WORK / GEOCML_NAME / 'EQA.dem_par'
layers = ('vel', 'vstd', 'coh_avg', 'n_unw')
exported = {}
for layer in layers:
    source = results_dir / layer
    if not source.exists():
        raise FileNotFoundError(f'Missing LiCSBAS result layer: {source}')
    destination = DRIVE_OUT / f'jakarta_{layer}_{DATE_TAG}_{RUN_TAG}.tif'
    run_script(
        'geotiff', ['-i', source, '-p', dem_par, '-o', destination],
        required=[destination],
    )
    exported[layer] = destination
print(json.dumps({k: str(v) for k, v in exported.items()}, indent=2))


In [ ]:
# 12 — Validate grids/values and write provenance plus a scientific QC summary
import numpy as np
import rasterio

def sha256(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

layer_meta = {}
grid_key = None
for role, path in exported.items():
    with rasterio.open(path) as src:
        data = src.read(1)
        current_grid = (src.width, src.height, str(src.crs), tuple(src.transform))
        if grid_key is None:
            grid_key = current_grid
        elif current_grid != grid_key:
            raise RuntimeError(f'Grid mismatch for {role}: {current_grid} != {grid_key}')
        finite = np.isfinite(data)
        values = data[finite]
        if role == 'vstd' and (not finite.any() or np.any(values < 0)):
            raise RuntimeError('vstd is empty or contains negative values')
        nodata_value = src.nodata
        if isinstance(nodata_value, float) and not np.isfinite(nodata_value):
            nodata_value = None
        layer_meta[role] = {
            'path': str(path), 'sha256': sha256(path), 'bytes': path.stat().st_size,
            'shape': [src.height, src.width], 'crs': str(src.crs),
            'transform': list(src.transform), 'resolution': list(src.res),
            'nodata': nodata_value,
            'total_pixels': int(data.size),
            'finite_pixels': int(finite.sum()),
            'finite_fraction': float(finite.mean()),
            'min': float(values.min()) if values.size else None,
            'median': float(np.median(values)) if values.size else None,
            'max': float(values.max()) if values.size else None,
        }

with rasterio.open(exported['vel']) as vel_src, rasterio.open(exported['vstd']) as vstd_src:
    common_valid = np.isfinite(vel_src.read(1)) & np.isfinite(vstd_src.read(1))

def count_data_lines(path):
    if path is None or not path.exists():
        return None
    return sum(1 for line in path.read_text(encoding='utf-8', errors='replace').splitlines() if line.strip() and not line.lstrip().startswith('#'))

info_dir = WORK / TS_NAME / 'info'
def first_info(pattern):
    matches = sorted(info_dir.glob(pattern)) if info_dir.exists() else []
    return matches[0] if matches else None

licsbas_network = {
    'step11_bad_interferograms': count_data_lines(first_info('11bad_ifg*.txt')),
    'step12_bad_interferograms': count_data_lines(first_info('12bad_ifg*.txt')),
    'step13_used_images': count_data_lines(first_info('13used_image*.txt')),
}
quality_flags = []
vel_meta = layer_meta['vel']
vstd_meta = layer_meta['vstd']
coh_meta = layer_meta['coh_avg']
n_unw_meta = layer_meta['n_unw']
if vel_meta['finite_fraction'] < 0.50:
    quality_flags.append('velocity coverage is below 50% of the output grid')
if coh_meta['median'] is not None and coh_meta['median'] < 0.20:
    quality_flags.append('median average coherence is below 0.20')
if vstd_meta['median'] is not None and vstd_meta['median'] >= 12.0:
    quality_flags.append('median vstd is close to or above the planned 15 mm/year threshold')
if vel_meta['min'] is not None and max(abs(vel_meta['min']), abs(vel_meta['max'])) > 200.0:
    quality_flags.append('absolute velocity extremes exceed 200 mm/year and require validation')
if n_unw_meta['max'] is not None and n_unw_meta['max'] < 20:
    quality_flags.append('maximum contributing unwrapped-interferogram count is below 20')

qc = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'scientific_status': 'diagnostic_candidate_pending_validation',
    'run_role': RUN_ROLE,
    'frame': FRAME, 'orbit': 'ascending', 'start': START, 'end': END,
    'multilook': NLOOK, 'run_tag': RUN_TAG, 'loop_mode': LOOP_MODE,
    'common_valid_vel_vstd_pixels': int(common_valid.sum()),
    'common_valid_vel_vstd_fraction': float(common_valid.mean()),
    'resolution': resolution_summary,
    'network': licsbas_network,
    'network_survival_gate': network_gate,
    'layer_statistics': layer_meta,
    'quality_flags': quality_flags,
    'interpretation': 'Structural completion is not scientific validation. Compare this factor-5 sensitivity result with the rejected factor-2 runs and the factor-10 regional diagnostic, then complete uncertainty, coherence, reference, and external validation before promotion.',
}
QC_SUMMARY.write_text(json.dumps(qc, indent=2, allow_nan=False), encoding='utf-8')

manifest = {
    'created_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'frame': FRAME, 'orbit': 'ascending', 'start': START, 'end': END,
    'multilook': NLOOK, 'run_tag': RUN_TAG, 'run_role': RUN_ROLE, 'loop_mode': LOOP_MODE,
    'licsbas_commit': actual_commit,
    'input_archive': {
        'path': str(GEOC_ARCHIVE), 'sha256': archive_sha,
        'complete_pairs': geoc_inventory['complete_pairs'],
    },
    'multilook_checkpoint': {
        'path': str(GEOCML_ARCHIVE), 'sha256': checkpoint_sha,
        'complete_pairs': geocml_inventory['complete_pairs'],
    },
    'resolution': resolution_summary,
    'network': licsbas_network,
    'network_survival_gate': network_gate,
    'steps': ['01', '02', '11', '12', '13', '14'],
    'layers': layer_meta,
    'qc_summary': str(QC_SUMMARY),
}
manifest_path = DRIVE_OUT / f'jakarta_licsbas_{DATE_TAG}_{RUN_TAG}_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8')
print(json.dumps(qc, indent=2))
print('QC summary:', QC_SUMMARY)
print('Manifest:', manifest_path)
if quality_flags:
    print('QC flags were recorded. The run is a diagnostic candidate, not a validated hazard layer.')


In [ ]:
# 13 — Archive final results and verify them
RESULTS_ARCHIVE = ARCHIVE / f'{TS_NAME}_results_{DATE_TAG}.tar.gz'
RESULTS_ARCHIVE_SHA = ARCHIVE / f'{TS_NAME}_results_{DATE_TAG}.sha256'
items = [f'{TS_NAME}/results', f'{TS_NAME}/info', f'{TS_NAME}/network', f'{GEOCML_NAME}/EQA.dem_par']
existing_items = [item for item in items if (WORK / item).exists()]
archive_run = subprocess.run(
    ['tar', '-czf', str(RESULTS_ARCHIVE), '-C', str(WORK), *existing_items]
)
if archive_run.returncode != 0 or not RESULTS_ARCHIVE.exists() or RESULTS_ARCHIVE.stat().st_size == 0:
    raise RuntimeError('Final-results archive failed')
subprocess.run(['tar', '-tzf', str(RESULTS_ARCHIVE)], check=True, stdout=subprocess.DEVNULL)
results_sha = sha256_file(RESULTS_ARCHIVE)
RESULTS_ARCHIVE_SHA.write_text(f'{results_sha}  {RESULTS_ARCHIVE.name}\n', encoding='utf-8')
print(f'VERIFIED final archive: {RESULTS_ARCHIVE} ({RESULTS_ARCHIVE.stat().st_size / 1e6:.1f} MB)')
print('Final archive SHA-256:', results_sha)
print(f'DONE: exact-date {RUN_TAG} GeoTIFFs, manifest, QC summary, and verified archives are in Drive.')
